In this section we will be using the Intrusion Detection Evaluation Dataset (CIC-IDS2017), a file of source-to-destination web traffic flows and testing how reliably Countmin Sketch can be used to identify DoS (Denial of Service) attacks.

Modern intrusion detection systems operate under significant practical constraints. Network traffic arrives as a high-volume stream, often consisting of millions of flows per hour, making it infeasible to store and process all data exactly. At the same time, detecting malicious behavior—such as coordinated attacks targeting a specific destination—requires identifying unusual patterns in near real time.

In this assignment, you will focus on detecting potential Denial of Service (DoS) behavior by analyzing source-to-destination communication patterns. Intuitively, DoS attacks often manifest as many requests originating from one or more sources and targeting a small set of destinations at unusually high frequencies.

However, instead of relying on exact counting methods (which can be memory-intensive and slow at scale), you will use a Count-Min Sketch, an approximate data structure that provides memory-efficient frequency estimates with probabilistic error guarantees.

In [ ]:
from __future__ import annotations

import hashlib
from dataclasses import dataclass
from typing import Dict, List, Mapping, MutableMapping, Set

import pandas as pd
from matplotlib import pyplot as plt
import numpy as np
import json
import sys
import warnings
warnings.filterwarnings("ignore")
# Load the data from Google Drive
from google.colab import drive
drive.mount('/content/drive/')  #you may be prompted to change the permissions on your google drive account here, just hit 'select all'




In [ ]:
CSV_PATH = (
)

df = pd.read_csv(
        CSV_PATH,
        low_memory=False,
    )
df.columns = [col.lstrip() for col in df.columns]
destips=df.groupby(by=['Destination IP'],as_index=False)['Label'].count()
destips=destips.sort_values(by="Label")
df['attack']=0
df.loc[df['Label']!="BENIGN",'attack']=1
attackers=df.groupby(by=['Source IP'])['attack'].max()



#TODO: Count the source-to-destination flow counts. Convert to a dictionary and compute the memory size of that dictionary

dictsize=None
print(dictsize)


In [ ]:

#TODO: Build a countmin sketch class that can be specified with varying widths and depths
class CountMinSketch:
    """Two-hash Count-Min Sketch for approximate frequency estimation."""

    def __init__(self, width: int = 2048,depth: int=2) -> None:

        #TODO: Fill in here, define the dimensions of the table

        self._seeds = ["dsthash"+str(k) for k in range(depth)]

    def _hash(self, key: str, row: int) -> int:
        payload = f"{self._seeds[row]}::{key}".encode("utf-8")
        digest = hashlib.blake2b(payload, digest_size=8).digest()
        return int.from_bytes(digest, byteorder="big", signed=False) % self.width

    def update(self, key: str, count: int = 1) -> None:
        #TODO: Fill in here, increase the value for all corresponding buckets

    def estimate(self, key: str) -> int:
        #TODO: Fill in here, retrieve the estimate that corresponds to a given key




def build_destination_sketches(
    width: int = 2048,
    chunksize: int = 100_000,
    tracked_sources_per_destination: int = 3,
    depth: int=2
) -> tuple[MutableMapping[str, CountMinSketch], BuildStats, Dict[str, Set[str]]]:
    """
    Build one Count-Min Sketch per destination IP.

    Each row contributes +1 to the destination sketch keyed by source IP.
    The CSV is streamed in chunks using Source IP, Destination IP, and Label columns.
    """
    required_columns = ["Source IP", "Destination IP", "Label"]

    sketches_by_destination: MutableMapping[str, CountMinSketch] = {}
    label_counts: Dict[str, int] = {}
    total_rows_processed = 0


    for src_ip, dst_ip, label in df[required_columns].itertuples(index=False, name=None):
        if pd.isna(src_ip) or pd.isna(dst_ip):
            continue

        src = str(src_ip).strip()
        dst = str(dst_ip).strip()
        if not src or not dst:
            continue

        sketch = sketches_by_destination.get(dst)
        if sketch is None:
            sketch = CountMinSketch(width=width,depth=depth)
            sketches_by_destination[dst] = sketch

        sketch.update(src, 1)
        total_rows_processed += 1

        label_key = "NaN" if pd.isna(label) else str(label).strip()
        label_counts[label_key] = label_counts.get(label_key, 0) + 1



    return sketches_by_destination




In [ ]:
#TODO: Generate the sketches for all the different destination addresses,
#First use a width of 10 and a depth of 2

#TODO: Get the memory size of *all* the countmin sketches. How does this compare to the pandas df
sketchSize=None

In [ ]:
##Going forward we will focus on the most contacted address. The server 192.168.10.50.
##TODO: Get the estimated flow counts for every IP address with a flow towards 192.168.10.50


##TODO: Get the Source IP addresses associated with the known attacks to server 192.168.10.50
attackSources=None
print(attackSources)

#TODO: How many unique attackers (Source IPs) were there on the server
NAttackers=None

In [ ]:
#TODO: Define over 1000 flows from a single source as an anomaly
#TODO: Create a histogram from the countmin sketch for destination IP 192.168.10.50 from all source IP addresses in the dataset.
##Put number of flows on the x axis, top code a bin for frequencies 1000 and higher (anomalies). Omit the range (0-100).
##Create a shaded background for the bars in which any attackers fall into.

bins = [100, 200, 300, 400,500,600,700,800,900,1000,1100]


#TODO: How many false negatives do you have?
FalseNeg=None
#TODO: How many false positives do you have?
FalsePos=None

What do your results in the previous graph suggest about the likelihood of a false negative when relying on countmin sketch in this way? How about the likelihood of a false negative?

In [ ]:
#TODO: Now create the countmin sketches with two more hash functions (4 total, same width as before)
##Create a histogram analogous to the other how many false positives do you have now?

FalsePos2=None


In [ ]:
##TODO:Choose width and depth such that your probability of a false positive is below 0.5%
##Treat a miscount of 1 (or larger) as a false positive.
FINALwidth=None
FINALdepth=None


In [ ]:
##TODO: Create a final CMS with the FINALwidth and FINALdepth. How many anomalies do you detect now?
##Optional: Make another histogram to visualize your CMS's values.
NAnomalies=None



In [ ]:
#JSON output
#Verify you haven't left any of these values as None
outputfile={
    "dictsize":dictsize,
    "sketchSize":sketchSize,
    "FalsePos":FalsePos,
        "FalseNeg":FalseNeg,
    "FalsePos2": FalsePos2,
    "FINALwidth": FINALwidth,
    "FINALdepth": FINALdepth,
    "NAnomalies":NAnomalies,
}
with open("CMS.json", "w") as f:
    json.dump(outputfile, f)